# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# If it fails to determine best cudnn convolution algorithm
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [ ]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

In [ ]:
# Disable all XLA auto-JIT compilation
tf.config.optimizer.set_jit(False)

# Spektral’s GCNConv uses a sparse-dense matmul under the hood.
# XLA’s GPU JIT compiler does not support that op.

### 1.3. GPU Management

In [ ]:
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 1000
EPOCHS = 100

DATA_SEED = 99
TRAIN_SEED = 111
SAMPLER_SEED = 111

# Set Python, NumPy, Keras and TensorFlow seeds
set_random_seed(TRAIN_SEED)

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
tf.config.experimental.enable_op_determinism()

In [ ]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

In [ ]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Hyperparameters

In [ ]:
kparams = KParams(
    activation_choices={
        "relu": tf.keras.activations.relu,
        "silu": tf.keras.activations.silu,  # Swish
        "elu": tf.keras.activations.elu,
        "gelu": tf.keras.activations.gelu,
        "tanh": tf.keras.activations.tanh,
        "none": None,
    },
    optimizer_choices={
        "lion": tf.keras.optimizers.Lion(beta_1=0.9, beta_2=0.99),
    },
    learning_rate=7e-5,
)

## 5. Model Definition

In [ ]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    from spektral.layers import GraphMasking, GlobalAvgPool

    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    A = build_knn_adjacency(rows=20, cols=200, k=trial.suggest_int("knn_k", 10, 16, step=2))

    x_graph, a_graph = GraphMasking()([combined, A])

    num_layers = trial.suggest_int("num_gnn_layers", 3, 5)
    for i in range(num_layers):
        # GNN layer with dropout
        x = build_cheb(
            trial=trial,
            kparams=kparams,
            x=x_graph if i == 0 else x,  # Use x_graph only for the first layer
            a_graph=a_graph,
            name_prefix=f"cheb_{i}",
            # Units
            units_range=(150, 400),
            units_step=10,
            # K: gains beyond 2 hops tend to be marginal in many tasks
            K_range=2,
            # Dropout
            dropout_rate_range=(0.0, 0.5),
            dropout_rate_step=0.05,
            # Other parameters
            kernel_initializer=initializer,
        )

    # Global pooling layer to reduce the graph to a fixed-size vector
    x = GlobalAvgPool(name="global_avg_pool")(x)

    # ———————————————————————————— Extra dense layers ———————————————————————————— #
    num_dense_layers = trial.suggest_int("num_dense_layers", 2, 4)

    for i in range(num_dense_layers):
        # Dense layer with dropout
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            # Units
            units_range=(250, 600),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.4),
            dropout_rate_step=0.2,
            # Other parameters
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,  # Disable XLA JIT compilation
    )

    return model

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    *,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    backup_dir = kwargs["backup_dir"]
    model_dir = kwargs["model_dir"]
    fig_dir = kwargs["fig_dir"]
    tensorboard_dir = kwargs["tensorboard_dir"]
    logs_dir = kwargs["logs_dir"]
    history_dir = kwargs["history_dir"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, kparams=kparams, show_summary=False)

        batch_size = 64
        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
            ),
            verbose=2,
        )

        trial.set_user_attr("best_train_accuracy", float(max(history.history.get("accuracy", []))))
        trial.set_user_attr("best_val_accuracy", float(max(history.history.get("val_accuracy", []))))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values) if DIRECTION == "minimize" else max(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bits_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
            batch_size=batch_size,
            n_trials=1000,
            verbose=True,
        )

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))
        trial.set_user_attr("test_loss_s009", float(test_loss))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss  # Value to minimize or maximize
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
        )

## Main

In [ ]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        pruner=optuna.pruners.HyperbandPruner(),
        sampler=optuna.samplers.TPESampler(seed=SAMPLER_SEED),
        load_if_exists=True,
        direction=DIRECTION,
    )

    study.optimize(
        lambda trial: objective(
            trial,
            epochs=EPOCHS,
            size_penalizer=None,
            **{
                "backup_dir": backup_dir,
                "model_dir": model_dir,
                "fig_dir": fig_dir,
                "logs_dir": logs_dir,
                "tensorboard_dir": tensorboard_dir,
                "history_dir": history_dir,
            },
        ),
        n_trials=get_remaining_trials(study, NUM_TRIALS),
        callbacks=[
            ImprovementStagnation(),
            StopIfKeepBeingPruned(threshold=50),
        ],
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,  # Use "value" for the study value
        order=ORDER,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
        (tensorboard_dir, "trial_{trial_id}"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
        "test_loss_s009",
        "test_loss_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))

except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)